Project 08 

In [3]:
#imports
#TODO: bash script filtering chr22 columns and only including columns 3 and 4 chromstart and end -> python dataframe for emission + transition probs
import math
import numpy as np

Initialization
Set up a probability matrix using the initial probabilities and the first observation.
Initialize a traceback matrix for path reconstruction.

Recursion
For each position and each possible state, calculate the maximum probability.
Store both the probabilities and the traceback pointers.
Apply transition and emission probabilities at each step.

Termination
Identify the final state with the highest probability.
Trace back through the matrix to reconstruct the optimal path.

# Considerations /edge cases:
# use neg log to prevent underflow
# Edge cases: handle zero probabilities, for example with pseudocounts.
#

In [15]:
class HiddenMarkovModel:
    """Hidden Markov Model for genomic sequence decoding."""

    def __init__(
        self,
        init_probs: dict[str, float],
        trans_probs: dict[str, dict[str, float]],
        emit_probs: dict[str, dict[str, float]],
    ) -> None:
        """
        Initialize the HMM object and set parameters.

        Args:
            init_probs: {state: P(start in state)}
            trans_probs: {from_state: {to_state: P(to_state | from_state)}}
            emit_probs: {state: {symbol: P(symbol | state)}}
        """
        self.init_probs = init_probs
        self.trans_probs = trans_probs
        self.emit_probs = emit_probs
        self.states = list(emit_probs.keys())

    def _viterbi_recursion(self, seq, prob_matrix, traceback):
        # Next position
        next_pos = len(prob_matrix)

        # Recursion end condition
        if next_pos == len(seq):
            return prob_matrix, traceback

        prev = prob_matrix[next_pos - 1]
        probs = {}
        pointers = {}
        for s in self.states:
            # Calculate maximum probability (best previous state p, maximizing score at t-1 + transition p -> s)
            best_prev = None
            best_score = float("-inf")
            for p in self.states:
                score = prev[p] + math.log(self.trans_probs[p][s])
                if score > best_score:
                    best_prev, best_score = p, score

            # Apply transition and emission probabilities at each step
            probs[s] = (
                prev[best_prev]
                + math.log(self.trans_probs[best_prev][s])
                + math.log(self.emit_probs[s][seq[next_pos]])
            )
            pointers[s] = best_prev

        # Store probability and traceback pointer
        prob_matrix.append(probs)
        traceback.append(pointers)

        return self._viterbi_recursion(seq, prob_matrix, traceback)

    def _viterbi_traceback(self, traceback, last_state):
        # Trace back through the matrix to reconstruct the optimal path
        path = [last_state]
        for t in range(len(traceback) - 1, 0, -1):
            path.append(traceback[t][path[-1]])
        path.reverse()
        return "".join(path)

    def viterbi(self, seq: str) -> tuple[str, float]:        
        # Initialization
        # Set up probability matrix using initial probabilities and the first observation
        # Initialize a traceback matrix for path reconstruction
        first_row = {}
        for s in self.states:
            first_row[s] = math.log(self.init_probs[s]) + math.log(self.emit_probs[s][seq[0]])
        prob_matrix = [first_row]

        traceback = [{s: None for s in self.states}]

        # Recursion
        prob_matrix, traceback = self._viterbi_recursion(seq, prob_matrix, traceback)

        # Identify the final state with the highest probability
        last_row = prob_matrix[-1]
        last_state = None
        for s in self.states:
            if last_state is None or last_row[s] > last_row[last_state]:
                last_state = s

        log_prob = prob_matrix[-1][last_state]

        # Find most likely path
        path = self._viterbi_traceback(traceback, last_state)

        return path, log_prob


In [16]:
# Test with Toy Dataset

# Example initial probabilities (probability of starting in each state)
init_probs = {
    "I": 0.2,
    "G": 0.8
}

# Example transition probabilities (probability of moving from one state to another)
trans_probs = {
    "I": {"I": 0.7, "G": 0.3},
    "G": {"I": 0.1, "G": 0.9}
}

# Example emission probabilities (probability of observing a symbol in a given state)
emit_probs = {
    "I": {"A": 0.1, "C": 0.4, "G": 0.4, "T": 0.1},
    "G": {"A": 0.3, "C": 0.2, "G": 0.2, "T": 0.3}
}

toy_hmm = HiddenMarkovModel(init_probs, trans_probs, emit_probs)
print(f"State path for input 'GGCACTGAA': {toy_hmm.viterbi("GGCACTGAA")}")
print(f"State path for input 'ATATCGCGCGCGAT': {toy_hmm.viterbi("ATATCGCGCGCGAT")}")

State path for input 'GGCACTGAA': ('GGGGGGGGG', -13.929108456051065)
State path for input 'ATATCGCGCGCGAT': ('GGGGIIIIIIIIGG', -21.20203079978548)


In [ ]:
## Driver Program with Chr 22 Dataset 


